Despues de hacer web scrapping, procedemos a hacer limpieza de los datos extraidos.. empezamos cargando las librerias correspondientes y convertir el csv en un df


In [1]:
import pandas as pd 

# 1. Carga de datos 
df = pd.read_csv('mazda3.csv')


Ahora realizamos un df.info() y df.head() para ver el formato de la info descargada mediante web scrapping, y si se necesita limpiar, depurar, etc..

In [2]:
df.info()

df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   detalles  168 non-null    object
 1   modelo    168 non-null    object
 2   precio    113 non-null    object
dtypes: object(3)
memory usage: 4.1+ KB


,detalles,modelo,precio
0,"2022 • 17,392 km • 2.0 MHEV I SPORT AUTO • Aut...",Mazda • Mazda 3,"$\n320,999"
1,"2022 • 35,149 km • 2.5 I AUTO • Automático",Mazda • Mazda 3,"$\n276,999"
2,"2022 • 53,233 km • 2.5 TURBO SIGNATURE 4WD AUT...",Mazda • Mazda 3,"$\n326,999"
3,"2017 • 58,273 km • 2.5 HATCHBACK S GRAND TOURI...",Mazda • Mazda 3,"$\n213,999"
4,"2018 • 122,213 km • 2.0 SEDAN I TM • Manual",Mazda • Mazda 3,"$\n176,999"
5,"2022 • 101,587 km • 2.5 I SPORT • Manual",Mazda • Mazda 3,"$\n278,999"
6,"2021 • 37,507 km • 2.5 I SPORT AUTO • Automático",Mazda • Mazda 3,"$\n287,999"
7,"2022 • 25,600 km • 2.5 I SPORT AUTO • Automático",Mazda • Mazda 3,"$\n322,999"
8,"2018 • 23,526 km • 2.5 SEDAN S TM • Manual",Mazda • Mazda 3,"$\n248,999"
9,"2015 • 35,851 km • 2.0 SEDAN I TA • Automático",Mazda • Mazda 3,"$\n200,999"


Notamos que la columna detalles tiene muchos datos dentro de, cosa que tenemos que separar para poder examinar correctamente los datos, estos son; año, kilometrake, cilindrada, versión, carrocería y transmision, la columna de modelo es irrelevante ya que todos son mazda 3 y el precio debe limpiarse para que no aparezca el salto de linea "\n" y convertirlos a enteros.

In [6]:
#Empezamos convirtiendo todas la info a minusculas en una nueva columna

df["temp"] = df["detalles"].str.lower()

#Extraemos el año

df["año"] = df["temp"].str.extract(r"(\d{4})").astype(int)  #Agregamos la columna ano, que tomara el string donde haya 4 numeros seguidos y lo convertirmos a dato entero

df['temp'] = df['temp'].str.replace(r'\d{4}', '', regex=True)  #Eliminamos ese mismo ano de la columna

#Extraemos kilometraje

df["kilometros"] = df["temp"].str.replace(',', '').str.extract(r'(\d+)\s*[Kk][Mm]').astype(int) #Creamos la columna kilometros, quitamos la , con el .replace y extraemos los numeros que esten en km mayusculas o minusculas, al final lo pasamos a int

df['temp'] = df['temp'].str.replace(r'[\d,]+\s*km', '', regex=True) #Eliminamos el kilometraje de la columna

#Extraemos cilindrada

df["motor"] = df["temp"].str.extract(r'(\d\.\d)').astype(float) #Agregamos la columna motor, utilizamos el .extract y le decimos que busque un valor decimal, \d es numero, \. un punto y nuevamente \d otro numero, osea un float..


df['temp'] = df['temp'].str.replace(r'\d\.\d', '', regex=True) # Lo eliminamos de la columna temp

#Extraemos la transmision

df['transmision'] = 'manual' # Agregamos la columna transmision y le ponemos manual por defecto

df.loc[df['temp'].str.contains('auto|ta|automática', na=False), 'transmision'] = 'automático' # Aqui buscamos dentro de la columna temp, si tiene la palabra automatica,auto o ta, lacambia por automatico, sino se queda en manual

df['temp'] = df['temp'].str.replace(r'\b(auto|automática|ta|manual|tm|automático)\b', '', regex=True).str.strip() #Eliminamos de la columna

#Extraemos la carroceria

# Creamos la columna carroceria

df['carroceria'] = 'sedán' # Creamos columna carroceria y por default sedan

df.loc[df['temp'].str.contains('hatchback|hb'), 'carroceria'] = 'hatchback' # Hacemos lo mismo que con automatico y manual, si encuentra  hatchback lo cambiara, sino no

df['temp'] = df['temp'].str.replace(r'\b(sedan|sedán|hatchback|hb)\b', '', regex=True).str.strip() #Eliminamos de temp

# Finalmente la version

df['version'] = df['temp'].str.replace(r'[•\-]', '', regex=True).str.strip() # Aqui reemplazamos puntos, guiones por un espacio vacio


df['version'] = df['version'].str.replace(r'\s+', ' ', regex=True) # Si quedaron espacios dobles, lo volvemos a un solo espacio

df.loc[df['version'] == 'i', 'version'] = 'i (base)' # Para que no salga solo i




Despues de toda esta limpieza por parte de la columna detalles, hay que revisar para ver que nos hace falta

In [7]:
df.head(10)

,detalles,modelo,precio,temp,año,kilometros,motor,transmision,carroceria,version
0,"2022 • 17,392 km • 2.0 MHEV I SPORT AUTO • Aut...",Mazda • Mazda 3,"$\n320,999",• • mhev i sport •,2022,17392,2.0,automático,sedán,mhev i sport
1,"2022 • 35,149 km • 2.5 I AUTO • Automático",Mazda • Mazda 3,"$\n276,999",• • i •,2022,35149,2.5,automático,sedán,i (base)
2,"2022 • 53,233 km • 2.5 TURBO SIGNATURE 4WD AUT...",Mazda • Mazda 3,"$\n326,999",• • turbo signature 4wd •,2022,53233,2.5,automático,sedán,turbo signature 4wd
3,"2017 • 58,273 km • 2.5 HATCHBACK S GRAND TOURI...",Mazda • Mazda 3,"$\n213,999",• • s grand touring •,2017,58273,2.5,automático,hatchback,s grand touring
4,"2018 • 122,213 km • 2.0 SEDAN I TM • Manual",Mazda • Mazda 3,"$\n176,999",• • i •,2018,122213,2.0,manual,sedán,i (base)
5,"2022 • 101,587 km • 2.5 I SPORT • Manual",Mazda • Mazda 3,"$\n278,999",• • i sport •,2022,101587,2.5,manual,sedán,i sport
6,"2021 • 37,507 km • 2.5 I SPORT AUTO • Automático",Mazda • Mazda 3,"$\n287,999",• • i sport •,2021,37507,2.5,automático,sedán,i sport
7,"2022 • 25,600 km • 2.5 I SPORT AUTO • Automático",Mazda • Mazda 3,"$\n322,999",• • i sport •,2022,25600,2.5,automático,sedán,i sport
8,"2018 • 23,526 km • 2.5 SEDAN S TM • Manual",Mazda • Mazda 3,"$\n248,999",• • s •,2018,23526,2.5,manual,sedán,s
9,"2015 • 35,851 km • 2.0 SEDAN I TA • Automático",Mazda • Mazda 3,"$\n200,999",• • i •,2015,35851,2.0,automático,sedán,i (base)


Hay varios temas que tratar, por ejemplo eliminar las columnas detalles y temp, ya no son necesarias, la columna modelo podemos eliminarla tambien si sabemos que es un db de Mazda 3 o podemos reducirlo al nombre del auto si tenemos mas marcas/modelos, hay que cambiar precio a un entero y en version ver cuantas variantes tenemos y revisar si hay alguno repetido

In [9]:
# Convertirmos el precio a int y eliminamos strings 

df["precio"] = df["precio"].str.replace(r"[\$\n,]", "", regex=True).astype("Int64")

#Eliminamos las columnas detalles y temp

df.drop(columns=["detalles", "temp", "modelo"], inplace=True)

#Cuantas versiones hay

print(df['version'].value_counts())

version
i sport                          43
i grand touring                  41
s grand touring                  21
i touring                        16
i (base)                         14
s                                12
turbo signature 4wd              11
mhev i sport                      4
turbo s grand touring 4wd         2
signature 4wd                     2
turbo signature polymetal 4wd     1
s gt                              1
Name: count, dtype: int64


In [10]:
# Diccionario de equivalencias para unir versiones similares

mapeo_versiones = {
    's gt': 's grand touring',
    'signature 4wd': 'turbo signature 4wd',
    'mhev i sport': 'i sport'
}

# Aplicamos el reemplazo

df['version'] = df['version'].replace(mapeo_versiones)

# Validamos nuevamente

print(df['version'].value_counts())


version
i sport                          47
i grand touring                  41
s grand touring                  22
i touring                        16
i (base)                         14
turbo signature 4wd              13
s                                12
turbo s grand touring 4wd         2
turbo signature polymetal 4wd     1
Name: count, dtype: int64


In [11]:
df.info()

df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   precio       113 non-null    Int64  
 1   año          168 non-null    int32  
 2   kilometros   168 non-null    int32  
 3   motor        168 non-null    float64
 4   transmision  168 non-null    object 
 5   carroceria   168 non-null    object 
 6   version      168 non-null    object 
dtypes: Int64(1), float64(1), int32(2), object(3)
memory usage: 8.2+ KB


,precio,año,kilometros,motor,transmision,carroceria,version
0,320999,2022,17392,2.0,automático,sedán,i sport
1,276999,2022,35149,2.5,automático,sedán,i (base)
2,326999,2022,53233,2.5,automático,sedán,turbo signature 4wd
3,213999,2017,58273,2.5,automático,hatchback,s grand touring
4,176999,2018,122213,2.0,manual,sedán,i (base)
5,278999,2022,101587,2.5,manual,sedán,i sport
6,287999,2021,37507,2.5,automático,sedán,i sport
7,322999,2022,25600,2.5,automático,sedán,i sport
8,248999,2018,23526,2.5,manual,sedán,s
9,200999,2015,35851,2.0,automático,sedán,i (base)


Finalmente hicimos la limpieza del df, ahora vamos a realizar graficas.

In [13]:
#Guardamos csv limpio 

df.to_csv('mazda3_limpio.csv', index=False)